In [3]:
import librosa
import IPython.display as ipd
import numpy as np
from scipy.signal import lfilter

In [4]:
filename = '../../data/tomsdiner_mono.wav'
diner, srate = librosa.load(filename)
ipd.Audio(diner, rate=srate)

## KARPLUS-STRONG STRING SYNTHESIS

Using the karplus-strong string synthesis technique and experimenting by exciting the string based on our single sound

In [5]:
# Synthesize a KS string using a sound clip instead of frequency
def ks_clip_resonator(duration, clip, sample_rate=44100, decay=0.996, normalize=True):
    ring_buffer = np.asarray(clip, dtype=float)
    ring_buffer = ring_buffer - ring_buffer.mean()

    num_samples = int(duration * sample_rate)
    audio = np.zeros(num_samples)
 
    for i in range(num_samples):
        audio[i] = ring_buffer[0]

        new_sample = decay * 0.5 * (ring_buffer[0] + ring_buffer[1])

        ring_buffer = np.roll(ring_buffer, -1)
        ring_buffer[-1] = new_sample
 
    if normalize and np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio))
    return audio

# Synthesize a KS string by passing diner in to constantly excite the string
def ks_filter(x, period, decay=0.996, blend=0.5, invert=False):
    sign = 1.0 if invert else -1.0
    a = np.zeros(period + 2)
    a[0] = 1.0
    a[period] = sign * decay * (1 - blend)
    a[period + 1] = sign * decay * blend
    return lfilter([1.0], a, x)

# Synthesize a KS string using a clip segment as the initial excitation (pluck) of the string
def pluck(excitation, freq, duration, sr, **kw):
    x = np.zeros(int(duration * sr))
    x[:len(excitation)] = excitation - np.mean(excitation)
    y = ks_filter(x, int(round(sr / freq - 0.5)), **kw)
    return y / np.max(np.abs(y))

# Found this online; apply feedback to each sample by delaying 
def ks_nonlinear(input, delay, decay=0.996):
    num_samples = len(input)
    audio = np.zeros(num_samples)
    for i in range(num_samples):
        a = audio[i - delay] if i >= delay else 0.0
        b = audio[i - delay - 1] if i >= delay + 1 else 0.0
        feedback = np.tanh(decay * (0.5 * a + 0.5 * b))
        audio[i] = input[i] + feedback
    return audio


clip_string = ks_clip_resonator(5, diner[10000:11000], sample_rate=srate)
full_excitation = ks_filter(diner, int(round(srate/440-0.5)))
full_excitation_inverted = ks_filter(diner, int(round(srate/440-0.5)), invert=True)
initial_excitation = pluck(diner[:5000], 440, 5, sr=srate)
initial_excitation_inverted = pluck(diner[10000:12000], 440, 5, sr=srate, invert=True)
feedback_string = ks_nonlinear(diner, 400)

ipd.display(ipd.Audio(clip_string, rate=srate))
ipd.display(ipd.Audio(full_excitation, rate=srate))
ipd.display(ipd.Audio(full_excitation_inverted, rate=srate))
ipd.display(ipd.Audio(initial_excitation, rate=srate))
ipd.display(ipd.Audio(initial_excitation_inverted, rate=srate))
ipd.display(ipd.Audio(feedback_string, rate=srate))